# 04 · Train the FP32 DeepPixBiS baseline

**QPAD — Calibration-Set Composition Bias in INT8 Face Presentation Attack Detection**
CS 228 Biometric Security with AI · San José State University · Ananya Tyagi & Vindhya Channarayapatna Ravishankar

**Priority:** 0 (Checkpoint 1)

**Purpose.** Train DeepPixBiS once on the Protocol II training split, select the epoch on development loss, and freeze it.

**In plain words.** The network looks at a 224×224 face and outputs a 14×14 grid of 'realness' values; training pushes the grid towards 1 for real faces and 0 for attacks. The development split only decides which epoch to keep. After this notebook the weights never change again — every INT8 model comes from this one checkpoint.

**Expected inputs**
- Processed train shards (train frames) and dev shards (evaluation frames).

**Expected outputs**
- `QPAD_Project/models/deeppixbis_best_fp32.pt`, `deeppixbis_last_fp32.pt`, `training_config.json`
- `QPAD_Project/results/training_history.csv`, `training_curves.png`
- `deeppixbis_best_fp32.pt.FROZEN.json` (SHA-256 of the frozen checkpoint)

**Safety rules.** The OULU-NPU dataset is access-restricted. This notebook never uploads data, never creates
share links, never prints credentials, and never writes videos or face crops to the Git repository.
Long operations run only after you change `DRY_RUN = True` to `False` and read the pre-flight summary.

In [ ]:
# --- Step 1: mount Google Drive (private storage) -------------------------------------------
# Authentication happens in Colab's own pop-up; nothing about it is stored in this notebook.
import os, sys, json, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.environ.get("QPAD_SKIP_DRIVE_MOUNT"):
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive mount skipped (not in Colab, or test mode).")

In [ ]:
# --- Step 2: make the QPAD code importable ------------------------------------------------------
# Option A (default): upload the whole `qpad/` repository folder to
#   MyDrive/QPAD_Project/code/qpad
# Option B: set REPO_URL to your PRIVATE GitHub repository; Colab/GitHub will ask you to authenticate.
#   Never paste access tokens into this notebook.
REPO_URL = ""  # e.g. "https://github.com/<account>/qpad.git"
QPAD_CODE_DIR = Path(os.environ.get("QPAD_CODE_DIR", "/content/drive/MyDrive/QPAD_Project/code/qpad"))
if REPO_URL:
    QPAD_CODE_DIR = Path("/content/qpad")
    if not (QPAD_CODE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(QPAD_CODE_DIR)], check=True)
if not (QPAD_CODE_DIR / "src" / "qpad").is_dir():
    raise FileNotFoundError(f"QPAD code not found at {QPAD_CODE_DIR}. Upload the repo folder or set REPO_URL.")
sys.path.insert(0, str(QPAD_CODE_DIR / "src"))

if IN_COLAB and not os.environ.get("QPAD_SKIP_PIP"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(QPAD_CODE_DIR / "requirements-colab.txt")], check=True)
import qpad
print("qpad", qpad.__version__, "from", QPAD_CODE_DIR)

In [ ]:
# --- Step 3: authoritative paths -------------------------------------------------------------------
# These are the ONLY data/output locations. pathlib handles any spaces in paths.
from qpad.config import ProjectPaths, load_config, get_logger, set_global_seed, disk_free_gb

DATASET_ROOT = Path("/content/drive/MyDrive/QPAD_Project/OULU_NPU/raw")
TRAIN_TAR = DATASET_ROOT / "Train_files-002.tar"
DEV_TAR = DATASET_ROOT / "Dev_files-001.tar"
TEST_TAR = DATASET_ROOT / "Test_files-003.tar"
METADATA_ZIP = DATASET_ROOT / "OULU-NPU-20261010T032906Z-1-004.zip"
PROJECT_DRIVE_ROOT = Path("/content/drive/MyDrive/QPAD_Project")
LOCAL_WORK_ROOT = Path("/content/qpad_work")

cfgs = load_config(QPAD_CODE_DIR / "configs")
paths = ProjectPaths.from_config(cfgs["dataset"])
if not paths.overrides and not cfgs["_overrides"]:  # YAML must agree with the constants above
    assert paths.dataset_root == DATASET_ROOT and paths.project_drive_root == PROJECT_DRIVE_ROOT
    assert paths.archive("train") == TRAIN_TAR and paths.archive("dev") == DEV_TAR
    assert paths.archive("test") == TEST_TAR and paths.archive("metadata") == METADATA_ZIP
    assert paths.local_work_root == LOCAL_WORK_ROOT
for o in cfgs["_overrides"]:
    print("WARNING (test mode):", o)
paths.ensure_output_dirs()
print(paths.describe())

In [ ]:
# --- Step 4: software, GPU and disk information ------------------------------------------------
from qpad.reproducibility import environment_info, git_commit
from qpad.nbutils import df_h, tqdm_progress, preflight

ENV_INFO = environment_info()
print(json.dumps(ENV_INFO, indent=1, default=str))
print("git:", git_commit(QPAD_CODE_DIR))
print(df_h(LOCAL_WORK_ROOT if LOCAL_WORK_ROOT.exists() else Path("/content") if Path("/content").exists() else Path.cwd()))

In [ ]:
# --- Step 5: run configuration and seeds ---------------------------------------------------------
DRY_RUN = True                 # change to False ONLY after the dry run looks correct
DRY_RUN_VIDEOS_PER_CLASS = 2   # per split and class, deterministic (first IDs)
FORCE_REBUILD = False          # True re-creates outputs that already exist
RETRY_FAILED = False           # True re-processes items recorded as failed
RANDOM_SEED = 42
DRY_RUN_MAX_BATCHES = 3          # dry run: a few batches, 1 epoch
FREEZE_CHECKPOINT = False        # set True only after inspecting the curves of the FULL run
if os.environ.get("QPAD_FORCE_DRY_RUN"):
    DRY_RUN = True
SEED_INFO = set_global_seed(RANDOM_SEED)
logger = get_logger("qpad.nb04", paths.logs / "nb04.log")
logger.info("DRY_RUN=%s seed=%s", DRY_RUN, RANDOM_SEED)
from qpad.pipeline import output_paths
OUT = output_paths(paths, DRY_RUN)
OUT.ensure_output_dirs()
print("Outputs of this run go to:", OUT.project_drive_root)

## 1. Model and training recipe
Reference: George & Marcel, *Deep Pixel-wise Binary Supervision for Face Presentation Attack Detection*, ICB 2019 (Idiap `bob.paper.deep_pix_bis_pad`). Architecture: the first 8 blocks of ImageNet DenseNet-161 → 1×1 conv + sigmoid (14×14 map) → linear + sigmoid (binary output). Loss 0.5·pixel BCE + 0.5·binary BCE. See README *Deviations* for what is our own choice.

In [ ]:
import torch
DEVICE = os.environ.get("QPAD_DEVICE") or ("cuda" if torch.cuda.is_available() else "cpu")
if os.environ.get("QPAD_TEST_NO_PRETRAINED"):
    cfgs["training"]["model"]["pretrained_imagenet"] = False   # test mode only: no weight download
if os.environ.get("QPAD_TEST_NUM_WORKERS"):
    cfgs["training"]["data"]["num_workers"] = int(os.environ["QPAD_TEST_NUM_WORKERS"])
tcfg = cfgs["training"]
print("device:", DEVICE)
if DEVICE == "cpu" and not DRY_RUN:
    raise RuntimeError("A full training run needs a GPU runtime (Runtime -> Change runtime type -> GPU).")
if tcfg["runtime"]["deterministic"]:
    print("NOTE: deterministic mode can be slower; some CUDA ops only warn if no deterministic version exists.")
print(json.dumps(tcfg, indent=1))

In [ ]:
from qpad.pipeline import processed_split_dir
preflight("DeepPixBiS FP32 training", int(tcfg["schedule"]["epochs"]),
          [processed_split_dir(paths, DRY_RUN, "train"), processed_split_dir(paths, DRY_RUN, "dev")],
          [OUT.models, OUT.results], LOCAL_WORK_ROOT, tcfg, tcfg["runtime"]["seed"], DRY_RUN,
          "full run: roughly 1-3 min per epoch on a T4/L4 for ~21,600 training frames")

## 2. Train (resumes automatically from `deeppixbis_last_fp32.pt`)

In [ ]:
from qpad.pipeline import run_training
TRAIN = run_training(paths, cfgs, DRY_RUN, DEVICE, QPAD_CODE_DIR,
                     max_batches=DRY_RUN_MAX_BATCHES if DRY_RUN else None,
                     epochs_override=1 if DRY_RUN else None, logger=logger,
                     progress_factory=lambda d: tqdm_progress(d))
display(TRAIN["history"])

In [ ]:
from IPython.display import Image
if TRAIN["figure"]:
    display(Image(filename=TRAIN["figure"]["png"], width=640)); print(TRAIN["figure"]["caption"])
s = TRAIN["state"]
print(f"best epoch {s.best_epoch} | best dev loss {s.best_dev_loss:.5f} | epochs run {s.epoch}")

## 3. Freeze the selected checkpoint
Freezing records the SHA-256 of `deeppixbis_best_fp32.pt`. Later notebooks refuse to run if the file changes, and training refuses to overwrite a frozen checkpoint.

In [ ]:
from qpad.training import freeze_checkpoint, BEST_NAME
best = OUT.models / BEST_NAME
if DRY_RUN:
    print("dry run: freezing the DRY-RUN checkpoint so notebook 05 can be dry-run:", freeze_checkpoint(best))
elif FREEZE_CHECKPOINT:
    print("FROZEN:", freeze_checkpoint(best, {"best_epoch": TRAIN["state"].best_epoch}))
else:
    print("Inspect the curves, then set FREEZE_CHECKPOINT = True and re-run this cell.")

In [ ]:
# --- Final summary -----------------------------------------------------------------------------
print("checkpoint:", OUT.models / BEST_NAME, "| frozen:", (OUT.models / (BEST_NAME + ".FROZEN.json")).is_file())
print("history   :", OUT.results / "training_history.csv")

**Next notebook:** `05_threshold_and_fp32_evaluation.ipynb`